# Guía de métodos de tensores

Cada método tiene **para qué sirve**, **qué significa cada argumento**, **cuándo usarlo**, sus **errores típicos** y un **ejemplo** que puedes ejecutar y cambiar.

**Cómo usarla**
1. Lee la explicación e intenta predecir la salida antes de ejecutar la celda.
2. Cambia los valores del ejemplo y vuelve a ejecutarla.
3. Cuando lo tengas claro, haz los ejercicios de `tensor_ops.py`.

> No existen `torch_ones` ni `torch_zeros`: se escribe con punto, `torch.ones` y `torch.zeros`.

**Cómo leer las firmas.** `torch.zeros(*size, dtype=None)` significa que `size` admite varios valores seguidos y que `dtype` es opcional. `x.flip(dims)` se llama sobre un tensor; en la mayoría de los casos, `x.sum()` y `torch.sum(x)` son lo mismo.

*\* El tipo por defecto de las funciones de creación es `float32`, salvo que se indique otra cosa.*

In [1]:
import numpy as np
import torch

torch.manual_seed(0)  # los números aleatorios se repiten en cada ejecución

---

# 1. Crear tensores con un valor

## `torch.zeros`

```python
torch.zeros(*size, dtype=None, device=None, requires_grad=False)
```

Crea un tensor de la forma indicada con todos sus valores a **0**.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `*size` | La forma: `torch.zeros(2, 3)` o `torch.zeros((2, 3))`, que son equivalentes. | obligatorio |
| `dtype` | Tipo de los datos, por ejemplo `torch.float32`, `torch.int64` o `torch.bool`. | `float32`* |
| `device` | Dónde se crea: `"cpu"` o `"cuda"`. | CPU |
| `requires_grad` | Si es `True`, autograd registra las operaciones sobre este tensor. | `False` |

**Cuándo usarlo:** para inicializar acumuladores, máscaras vacías o el fondo de una imagen. Por ejemplo, `out = torch.zeros(B, C, H, W)` antes de rellenar en un bucle.

**⚠️ Cuidado:** si ya tienes un tensor de referencia, usa `zeros_like` o `new_zeros`; así no te olvidas de `dtype` ni de `device`.


In [2]:
print(torch.zeros(2, 3))                       # 2 filas y 3 columnas
print(torch.zeros((2, 3)))                     # lo mismo, con una tupla
print(torch.zeros(3, dtype=torch.int64))       # enteros
print(torch.zeros(2, 2, dtype=torch.bool))     # False

tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([0, 0, 0])
tensor([[False, False],
        [False, False]])


## `torch.ones`

```python
torch.ones(*size, dtype=None, device=None, requires_grad=False)
```

Igual que `zeros`, pero con todos los valores a **1**.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `*size` | La forma: `torch.zeros(2, 3)` o `torch.zeros((2, 3))`, que son equivalentes. | obligatorio |
| `dtype` | Tipo de los datos, por ejemplo `torch.float32`, `torch.int64` o `torch.bool`. | `float32`* |
| `device` | Dónde se crea: `"cpu"` o `"cuda"`. | CPU |

**Cuándo usarlo:** para máscaras en las que todo es válido y como objetivo de las imágenes reales en una GAN: `torch.ones(B)`.


In [3]:
print(torch.ones(2, 3))
print(torch.ones(2, 2, dtype=torch.bool))   # True en todas las posiciones
print(torch.ones(3) * 5)                    # todo a 5, aunque torch.full es más claro

tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[True, True],
        [True, True]])
tensor([5., 5., 5.])


## `torch.full`

```python
torch.full(size, fill_value, dtype=None, device=None)
```

Crea un tensor con **todos los valores iguales al que elijas**.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `size` | La forma. **Aquí tiene que ser una tupla o lista**. | obligatorio |
| `fill_value` | Valor con el que se rellena. | obligatorio |
| `dtype` | Tipo de los datos. Si se omite, se deduce de `fill_value`. | deducido |
| `device` | Dónde se crea: `"cpu"` o `"cuda"`. | CPU |

**Cuándo usarlo:** para imágenes constantes, como la *baseline* de Integrated Gradients o un valor de `ignore_index`.

**⚠️ Cuidado:** `torch.full(2, 3)` da error: la forma debe ir como `(2, 3)`.


In [4]:
print(torch.full((2, 3), 7.0))                     # float32
print(torch.full((2, 3), 7))                       # int64: lo deduce del 7
print(torch.full((2,), 7, dtype=torch.float32))    # forzamos float

tensor([[7., 7., 7.],
        [7., 7., 7.]])
tensor([[7, 7, 7],
        [7, 7, 7]])
tensor([7., 7.])


## `torch.eye`

```python
torch.eye(n, m=None, dtype=None, device=None)
```

Crea una matriz **identidad**: unos en la diagonal y ceros en el resto.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `n` | Número de filas. | obligatorio |
| `m` | Número de columnas. | igual que `n` |
| `dtype` | Tipo de los datos, por ejemplo `torch.float32`, `torch.int64` o `torch.bool`. | `float32`* |

**Cuándo usarlo:** para inicializar transformaciones, o como *one-hot*: `torch.eye(num_classes)[labels]`.


In [5]:
print(torch.eye(3))
print(torch.eye(2, 3))   # matriz rectangular

tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
tensor([[1., 0., 0.],
        [0., 1., 0.]])


## `torch.empty`

```python
torch.empty(*size, dtype=None, device=None)
```

Reserva memoria **sin inicializarla**: los valores son lo que hubiera en esa memoria.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `*size` | La forma: `torch.zeros(2, 3)` o `torch.zeros((2, 3))`, que son equivalentes. | obligatorio |
| `dtype` | Tipo de los datos, por ejemplo `torch.float32`, `torch.int64` o `torch.bool`. | `float32`* |

**Cuándo usarlo:** cuando vas a rellenarlo inmediatamente, por ejemplo `torch.empty_like(x).uniform_(-eps, eps)` en PGD.

**⚠️ Cuidado:** nunca uses sus valores directamente: no son ceros.


In [6]:
x = torch.empty(2, 3)
print(x.shape)   # no mires los valores: son basura
x.uniform_(-1, 1)   # así se rellena en el sitio
print(x)

torch.Size([2, 3])
tensor([[-0.0075,  0.5364, -0.8230],
        [-0.7359, -0.3852,  0.2682]])


---

# 2. Crear secuencias

## `torch.arange`

```python
torch.arange(start=0, end, step=1, dtype=None, device=None)
```

Crea una secuencia con un **paso fijo**. Igual que `range` de Python, **no incluye `end`**.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `start` | Primer valor. | 0 |
| `end` | Límite, sin incluirlo. | obligatorio |
| `step` | Distancia entre valores consecutivos. Puede ser negativa. | 1 |
| `dtype` | Tipo; se deduce: enteros si todos los argumentos lo son. | deducido |

**Cuándo usarlo:** cuando sabes el **paso**. Es lo habitual para generar índices.


In [7]:
print(torch.arange(5))            # solo end
print(torch.arange(2, 5))         # start y end
print(torch.arange(0, 10, 2))     # con paso
print(torch.arange(5, 0, -1))     # hacia atrás
print(torch.arange(0, 1, 0.25))   # float, sin incluir el 1

# Uso clásico: un elemento por fila
scores = torch.tensor([[0.1, 0.9], [0.8, 0.2]])
labels = torch.tensor([1, 0])
print(scores[torch.arange(2), labels])

tensor([0, 1, 2, 3, 4])
tensor([2, 3, 4])
tensor([0, 2, 4, 6, 8])
tensor([5, 4, 3, 2, 1])
tensor([0.0000, 0.2500, 0.5000, 0.7500])
tensor([0.9000, 0.8000])


## `torch.linspace`

```python
torch.linspace(start, end, steps, dtype=None, device=None)
```

Crea `steps` valores equiespaciados. **Incluye `end`**.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `start` | Primer valor. | obligatorio |
| `end` | Último valor, incluido. | obligatorio |
| `steps` | **Cuántos valores** quieres, no el tamaño del paso. | obligatorio |
| `dtype` | Tipo de los datos, por ejemplo `torch.float32`, `torch.int64` o `torch.bool`. | `float32`* |

**Cuándo usarlo:** cuando sabes **cuántos** valores quieres: umbrales de mAP o los pasos de Integrated Gradients.

**⚠️ Cuidado:** `steps` no es el paso: `linspace(0, 1, 5)` da 5 valores, con un paso de 0.25.


In [8]:
print(torch.linspace(0, 1, 5))
print(torch.linspace(0, 1, 3))

# Interpolación de Integrated Gradients
print(torch.linspace(0, 1, 5).view(-1, 1, 1, 1).shape)

tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
tensor([0.0000, 0.5000, 1.0000])
torch.Size([5, 1, 1, 1])


## `torch.logspace`

```python
torch.logspace(start, end, steps, base=10.0, dtype=None)
```

Valores equiespaciados en **escala logarítmica**: desde `base**start` hasta `base**end`.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `start` | Exponente inicial. | obligatorio |
| `end` | Exponente final. | obligatorio |
| `steps` | Número de valores. | obligatorio |
| `base` | Base de la potencia. | 10.0 |

**Cuándo usarlo:** para explorar hiperparámetros, por ejemplo *learning rates* o la razón α/β de *style transfer*.


In [9]:
print(torch.logspace(-3, 0, 4))   # 0.001, 0.01, 0.1 y 1

tensor([0.0010, 0.0100, 0.1000, 1.0000])


---

# 3. Crear tensores aleatorios

Fija `torch.manual_seed(n)` si necesitas que salgan siempre los mismos números.

## `torch.rand`

```python
torch.rand(*size, dtype=None, device=None)
```

Valores de una distribución **uniforme en [0, 1)**.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `*size` | La forma: `torch.zeros(2, 3)` o `torch.zeros((2, 3))`, que son equivalentes. | obligatorio |
| `dtype` | Tipo de los datos, por ejemplo `torch.float32`, `torch.int64` o `torch.bool`. | `float32`* |

**Cuándo usarlo:** para probabilidades, máscaras aleatorias como las de RISE y para inicializar imágenes.


In [10]:
print(torch.rand(2, 2))

# Uniforme en [a, b)
a, b = -3, 3
print((b - a) * torch.rand(3) + a)

# Máscara con un 30 % de True
print(torch.rand(6) < 0.3)

tensor([[0.4901, 0.8964],
        [0.4556, 0.6323]])
tensor([-0.9066, -0.5897, -2.8660])
tensor([ True,  True, False, False, False,  True])


## `torch.randn`

```python
torch.randn(*size, dtype=None, device=None)
```

Valores de una distribución **normal de media 0 y desviación 1**.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `*size` | La forma: `torch.zeros(2, 3)` o `torch.zeros((2, 3))`, que son equivalentes. | obligatorio |
| `dtype` | Tipo de los datos, por ejemplo `torch.float32`, `torch.int64` o `torch.bool`. | `float32`* |

**Cuándo usarlo:** para el vector latente `z` de GAN y VAE, y para el ruido de SmoothGrad.


In [11]:
print(torch.randn(2, 2))

# Normal de media 5 y desviación 2
print(5 + 2 * torch.randn(3))

tensor([[-0.0451,  0.8797],
        [ 1.0482, -0.0445]])
tensor([ 3.5542, 10.7327,  3.8690])


## `torch.randint`

```python
torch.randint(low=0, high, size, dtype=torch.int64, device=None)
```

Enteros aleatorios en **[low, high)**.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `low` | Valor mínimo, incluido. | 0 |
| `high` | Valor máximo, **sin incluir**. | obligatorio |
| `size` | La forma. **Debe ser una tupla**: `(4,)` para 1D. | obligatorio |

**Cuándo usarlo:** para etiquetas de prueba, coordenadas al azar como las de CutMix y *crops*.

**⚠️ Cuidado:** `torch.randint(0, 10, 4)` da error: hace falta `(4,)`.


In [12]:
print(torch.randint(10, (4,)))         # de 0 a 9
print(torch.randint(5, 10, (2, 3)))    # de 5 a 9
print(torch.randint(0, 2, (5,)))       # 0 o 1

tensor([5, 2, 9, 1])
tensor([[8, 8, 8],
        [6, 9, 6]])
tensor([1, 1, 1, 0, 1])


## `torch.randperm`

```python
torch.randperm(n, dtype=torch.int64, device=None)
```

Una **permutación aleatoria** de los enteros de 0 a `n - 1`.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `n` | Número de elementos. | obligatorio |

**Cuándo usarlo:** para barajar datos, en Mixup, Monte Carlo Shapley o el test de etiquetas aleatorias.


In [13]:
print(torch.randperm(5))

x = torch.tensor([10, 20, 30, 40])
print(x[torch.randperm(4)])   # baraja x

tensor([0, 2, 4, 1, 3])
tensor([40, 10, 20, 30])


## `torch.bernoulli`

```python
torch.bernoulli(input)
```

Muestrea 0 o 1 en cada posición. `input` contiene la **probabilidad de obtener 1**.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `input` | Tensor `float` con valores en [0, 1]. | obligatorio |

**Cuándo usarlo:** para Dropout hecho a mano y para muestrear píxeles binarios en PixelCNN.


In [14]:
probabilities = torch.tensor([0.0, 0.5, 1.0])
print(torch.bernoulli(probabilities))
print(torch.bernoulli(torch.full((5,), 0.8)))

tensor([0., 1., 1.])
tensor([1., 1., 1., 1., 1.])


## `torch.multinomial`

```python
torch.multinomial(input, num_samples, replacement=False)
```

Muestrea **índices** según unos pesos.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `input` | Pesos no negativos. No hace falta que sumen 1. | obligatorio |
| `num_samples` | Cuántos índices quieres. | obligatorio |
| `replacement` | Si es `True`, un índice puede repetirse. | `False` |

**Cuándo usarlo:** para muestrear de `softmax(logits)`, por ejemplo el siguiente píxel en PixelCNN.

**⚠️ Cuidado:** con `replacement=False`, no puedes pedir más muestras que índices con peso positivo.


In [15]:
weights = torch.tensor([0.1, 0.1, 0.8])
print(torch.multinomial(weights, 2))
print(torch.multinomial(weights, 5, replacement=True))

tensor([2, 0])
tensor([1, 2, 2, 2, 2])


---

# 4. Crear a partir de otro tensor

Evitan errores de `dtype` y de `device`, porque copian los del tensor de referencia.

## `torch.zeros_like / torch.ones_like`

```python
torch.zeros_like(input, dtype=None, device=None)
torch.ones_like(input, dtype=None, device=None)
```

Crean ceros o unos con **la misma forma, tipo y dispositivo** que `input`. No copian sus valores.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `input` | Tensor de referencia. | obligatorio |
| `dtype` | Por si quieres otro tipo. | el de `input` |

**Cuándo usarlo:** cuando el resultado tenga la misma forma: objetivos de pérdida o máscaras por píxel.


In [16]:
x = torch.tensor([[1.5, 2.5, 3.5]], dtype=torch.float64)
zeros = torch.zeros_like(x)
ones = torch.ones_like(x)
print(zeros, zeros.dtype)
print(ones, ones.dtype)
print(torch.ones_like(x, dtype=torch.bool))

# Los objetivos de una GAN
logits = torch.tensor([0.3, -0.8])
print(torch.ones_like(logits), torch.zeros_like(logits))

tensor([[0., 0., 0.]], dtype=torch.float64) torch.float64
tensor([[1., 1., 1.]], dtype=torch.float64) torch.float64
tensor([[True, True, True]])
tensor([1., 1.]) tensor([0., 0.])


## `torch.full_like`

```python
torch.full_like(input, fill_value, dtype=None)
```

Igual que `full`, pero copia forma, tipo y dispositivo de `input`.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `input` | Tensor de referencia. | obligatorio |
| `fill_value` | Valor de relleno. | obligatorio |

**Cuándo usarlo:** para una *baseline* gris en explicabilidad: `torch.full_like(image, 0.5)`.

**⚠️ Cuidado:** si `input` es entero, el valor se trunca: 7.9 pasa a 7.


In [17]:
x = torch.zeros(2, 2)
print(torch.full_like(x, 0.5))

# Cuidado: se convierte al tipo de input
print(torch.full_like(torch.tensor([1, 2]), 7.9))

tensor([[0.5000, 0.5000],
        [0.5000, 0.5000]])
tensor([7, 7])


## `torch.rand_like / torch.randn_like`

```python
torch.rand_like(input, dtype=None)
torch.randn_like(input, dtype=None)
```

Valores aleatorios con la forma de `input`: uniformes o normales.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `input` | Tensor de referencia. Debe ser `float`. | obligatorio |

**Cuándo usarlo:** para SmoothGrad y VAE (`eps = torch.randn_like(std)`).

**⚠️ Cuidado:** fallan con un tensor entero, salvo que indiques un `dtype` `float`.


In [18]:
image = torch.zeros(1, 2, 2)
print(torch.rand_like(image))
print(image + 0.1 * torch.randn_like(image))   # añade ruido

tensor([[[0.5995, 0.0652],
         [0.5460, 0.1872]]])
tensor([[[ 0.0160, -0.0040],
         [ 0.0288, -0.0008]]])


## `x.new_zeros / new_ones / new_full / new_tensor`

```python
x.new_zeros(size, dtype=None)
x.new_ones(size, dtype=None)
x.new_full(size, fill_value)
x.new_tensor(data)
```

Crean un tensor con **otra forma**, pero con el `dtype` y el `device` de `x`. Esa es la diferencia con `*_like`.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `size` | La forma del nuevo tensor. | obligatorio |
| `fill_value` | Valor de relleno, en `new_full`. | obligatorio |
| `data` | Datos, en `new_tensor`. | obligatorio |

**Cuándo usarlo:** dentro de un `forward`, para que el código funcione igual en CPU y GPU.


In [19]:
x = torch.ones(2, 2, dtype=torch.float64)
print(x.new_zeros(3).dtype)
print(x.new_ones(1, 2))
print(x.new_full((2,), 4.0))
print(x.new_tensor([1, 2]))

# Límites de la imagen para recortar cajas
boxes = torch.tensor([[-1.0, 5.0, 40.0, 60.0]])
print(boxes.new_tensor([30, 20, 30, 20]))

torch.float64
tensor([[1., 1.]], dtype=torch.float64)
tensor([4., 4.], dtype=torch.float64)
tensor([1., 2.], dtype=torch.float64)
tensor([30., 20., 30., 20.])


---

# 5. Convertir datos

## `torch.tensor`

```python
torch.tensor(data, dtype=None, device=None, requires_grad=False)
```

Crea un tensor **copiando** los datos de una lista, un número o un array.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `data` | Listas (anidadas), un número o un array de NumPy. | obligatorio |
| `dtype` | Si se omite: `float32` para decimales, `int64` para enteros y `bool` para booleanos. | deducido |
| `requires_grad` | Si necesita gradiente. Solo con tipos `float`. | `False` |

**Cuándo usarlo:** para escribir datos a mano, sobre todo en tests.

**⚠️ Cuidado:** `torch.tensor` y `torch.Tensor` son distintos: `torch.Tensor(3)` crea 3 valores sin inicializar. Usa siempre la minúscula.


In [20]:
print(torch.tensor([1, 2, 3]).dtype)
print(torch.tensor([1.0, 2.0]).dtype)
print(torch.tensor([True, False]).dtype)
print(torch.tensor(3.5))   # 0 dimensiones
print(torch.tensor([1, 2], dtype=torch.float32))
print(torch.tensor([[1, 2], [3, 4]]).shape)

torch.int64
torch.float32
torch.bool
tensor(3.5000)
tensor([1., 2.])
torch.Size([2, 2])


## `torch.from_numpy / x.numpy`

```python
torch.from_numpy(ndarray)
x.numpy()
```

Convierten entre NumPy y PyTorch **compartiendo memoria**: al cambiar uno, cambia el otro.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `ndarray` | Array de NumPy. | obligatorio |

**Cuándo usarlo:** al convertir imágenes: `torch.from_numpy(img).permute(2, 0, 1).float() / 255`.

**⚠️ Cuidado:** los arrays de NumPy suelen ser `float64`, y los modelos, `float32`: usa `.float()`. `x.numpy()` falla si `x` tiene `requires_grad=True`: antes hay que llamar a `.detach()`; si está en GPU, también a `.cpu()`.


In [21]:
array = np.array([1.0, 2.0])
shared = torch.from_numpy(array)
copied = torch.tensor(array)
array[0] = 99
print(shared)   # cambia
print(copied)   # no cambia
print(shared.dtype)   # NumPy usa float64

grad = torch.ones(2, requires_grad=True)
print(grad.detach().numpy())

tensor([99.,  2.], dtype=torch.float64)
tensor([1., 2.], dtype=torch.float64)
torch.float64
[1. 1.]


## `x.item / x.tolist`

```python
x.item()
x.tolist()
```

Convierten a tipos de Python. `item` solo funciona con **un único elemento**; `tolist` funciona con cualquier forma.

**Cuándo usarlo:** `total += loss.item()` en el bucle de entrenamiento.

**⚠️ Cuidado:** si acumulas `total += loss` sin `.item()`, conservas todo el grafo y crece la memoria.


In [22]:
x = torch.tensor([[1, 2], [3, 4]])
print(x[0, 0].item(), type(x[0, 0].item()))
print(x.tolist())

loss = torch.tensor(0.25, requires_grad=True)
print(loss.item())   # no retiene el grafo

1 <class 'int'>
[[1, 2], [3, 4]]
0.25


---

# 6. Inspeccionar

## `Atributos de un tensor`

```python
x.shape      x.size(dim)      x.ndim      x.numel()
x.dtype      x.device         x.requires_grad
```

Dan información sobre el tensor. Imprimir `x.shape` es la forma más rápida de depurar.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dim` | En `size`, la dimensión que quieres consultar. Sin argumento, devuelve toda la forma. | todas |


In [23]:
x = torch.zeros(2, 3, 4)
print("shape:", x.shape)
print("shape[0]:", x.shape[0])
print("size(-1):", x.size(-1))
print("ndim:", x.ndim)
print("numel:", x.numel())
print("dtype:", x.dtype)
print("device:", x.device)
print("requires_grad:", x.requires_grad)

b, c, h = x.shape   # desempaquetar
print(b, c, h)

shape: torch.Size([2, 3, 4])
shape[0]: 2
size(-1): 4
ndim: 3
numel: 24
dtype: torch.float32
device: cpu
requires_grad: False
2 3 4


---

# 7. Cambiar el tipo

## `x.float / long / bool / to`

```python
x.float()    x.long()    x.bool()
x.to(dtype_or_device, non_blocking=False)
```

Crean un tensor con otro tipo o en otro dispositivo. `float` es `float32`, y `long` es `int64`. Con `bool`, todo lo distinto de 0 pasa a `True`.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dtype_or_device` | En `to`, un tipo (`torch.uint8`), un dispositivo (`"cuda"`) u otro tensor del que copiar ambos. | obligatorio |

**Cuándo usarlo:** `.long()` para las etiquetas de `cross_entropy`, y `.float()` antes de `mean`.

**⚠️ Cuidado:** `.to()` **no** modifica el original: debes escribir `x = x.to(device)`. En cambio, `model.to(device)` sí modifica el modelo.


In [24]:
x = torch.tensor([0, 1, 2])
print(x.float())
print(x.bool())
print(torch.tensor([1.9, -1.9]).long())   # trunca hacia 0
print(x.to(torch.uint8).dtype)
print(x.to(torch.zeros(1).dtype).dtype)

# División: / siempre da float, y // es entera
print(x / 2, x // 2)

try:
    x.mean()
except RuntimeError as error:
    print("Error:", str(error)[:55])
print(x.float().mean())

tensor([0., 1., 2.])
tensor([False,  True,  True])
tensor([ 1, -1])
torch.uint8
torch.float32
tensor([0.0000, 0.5000, 1.0000]) tensor([0, 0, 1])
Error: mean(): could not infer output dtype. Input dtype must 
tensor(1.)


---

# 8. Cambiar la forma

## `x.reshape / x.view`

```python
x.reshape(*shape)
x.view(*shape)
```

Reordenan los mismos datos en otra forma; el número de elementos no puede cambiar. `-1` significa «calcula esta dimensión». `view` nunca copia y **falla** si la memoria no es contigua. `reshape` copia si hace falta.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `*shape` | La nueva forma. Puede tener un solo `-1`. | obligatorio |

**Cuándo usarlo:** `x.reshape(B, -1)` para aplanar. En caso de duda, usa `reshape`.

**⚠️ Cuidado:** los datos no cambian de orden: para intercambiar dimensiones hay que usar `permute`, no `reshape`.


In [25]:
x = torch.arange(6)
print(x.reshape(2, 3))
print(x.reshape(3, -1))
print(x.view(-1, 2))

# view falla tras permute
y = torch.arange(6).reshape(2, 3).permute(1, 0)
try:
    y.view(-1)
except RuntimeError as error:
    print("view falla:", str(error)[:45])
print(y.reshape(-1))
print(y.contiguous().view(-1))

tensor([[0, 1, 2],
        [3, 4, 5]])
tensor([[0, 1],
        [2, 3],
        [4, 5]])
tensor([[0, 1],
        [2, 3],
        [4, 5]])
view falla: view size is not compatible with input tensor
tensor([0, 3, 1, 4, 2, 5])
tensor([0, 3, 1, 4, 2, 5])


## `x.flatten`

```python
x.flatten(start_dim=0, end_dim=-1)
```

Une en una sola las dimensiones entre `start_dim` y `end_dim`, ambas incluidas.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `start_dim` | Primera dimensión que se une. | 0 |
| `end_dim` | Última dimensión que se une. | -1 |

**Cuándo usarlo:** `x.flatten(1)` tras las convoluciones, antes de una capa lineal.


In [26]:
x = torch.zeros(2, 3, 4, 5)
print(x.flatten().shape)
print(x.flatten(1).shape)      # conserva el batch
print(x.flatten(2).shape)      # (B, C, H*W)
print(x.flatten(1, 2).shape)

torch.Size([120])
torch.Size([2, 60])
torch.Size([2, 3, 20])
torch.Size([2, 12, 5])


## `x.permute / x.transpose / x.movedim`

```python
x.permute(*dims)
x.transpose(dim0, dim1)
x.movedim(source, destination)
```

**Reordenan dimensiones** sin copiar datos. `permute` indica el orden nuevo completo: cada número dice qué dimensión antigua va en esa posición. `transpose` intercambia dos. `movedim` mueve una.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `*dims` | Nuevo orden, con todas las dimensiones. | obligatorio |
| `dim0, dim1` | Las dos dimensiones que se intercambian. | obligatorio |
| `source, destination` | De dónde a dónde se mueve la dimensión. | obligatorio |

**Cuándo usarlo:** para pasar `(H, W, C)` a `(C, H, W)`; `k.transpose(-2, -1)` en atención.

**⚠️ Cuidado:** no copian memoria: al modificar el resultado cambia el original.


In [27]:
image = torch.zeros(32, 64, 3)        # (H, W, C)
print(image.permute(2, 0, 1).shape)   # (C, H, W)
print(image.movedim(-1, 0).shape)     # lo mismo

x = torch.zeros(2, 3, 4)
print(x.transpose(1, 2).shape)
print(x.transpose(-2, -1).shape)      # habitual en atención

m = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(m.permute(1, 0))

torch.Size([3, 32, 64])
torch.Size([3, 32, 64])
torch.Size([2, 4, 3])
torch.Size([2, 4, 3])
tensor([[1, 4],
        [2, 5],
        [3, 6]])


## `x.unsqueeze / x.squeeze / x[None]`

```python
x.unsqueeze(dim)
x.squeeze(dim=None)
```

`unsqueeze` **añade** una dimensión de tamaño 1. `squeeze` **quita** las dimensiones de tamaño 1. `x[None]` es un atajo para añadir una al principio.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dim` | En `unsqueeze`, la posición de la nueva dimensión. En `squeeze`, la dimensión que se quita (si vale 1). Sin indicarla, `squeeze` las quita todas. | `unsqueeze`: obligatorio |

**Cuándo usarlo:** para añadir el *batch* a una imagen (`model(image.unsqueeze(0))`) y preparar *broadcasting*.

**⚠️ Cuidado:** con un *batch* de tamaño 1, `squeeze()` sin argumento también elimina esa dimensión. Es más seguro indicar `dim`.


In [28]:
x = torch.zeros(3, 4)
print(x.unsqueeze(0).shape)
print(x.unsqueeze(-1).shape)
print(x[None].shape)
print(x[:, None].shape)
print(x[:, :, None].shape)

y = torch.zeros(1, 3, 1, 4)
print(y.squeeze().shape)
print(y.squeeze(0).shape)
print(y.squeeze(1).shape)   # no vale 1: no hace nada

# Normalización por canal
image = torch.rand(3, 4, 4)
mean = torch.tensor([0.5, 0.4, 0.3])
print((image - mean[:, None, None]).shape)

torch.Size([1, 3, 4])
torch.Size([3, 4, 1])
torch.Size([1, 3, 4])
torch.Size([3, 1, 4])
torch.Size([3, 4, 1])
torch.Size([3, 4])
torch.Size([3, 1, 4])
torch.Size([1, 3, 1, 4])
torch.Size([3, 4, 4])


---

# 9. Repetir

## `x.repeat`

```python
x.repeat(*sizes)
```

Repite el tensor **copiando datos**. Cada número dice cuántas veces se repite en esa dimensión.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `*sizes` | Repeticiones por dimensión. Si hay más números que dimensiones, se añaden dimensiones al principio. | obligatorio |

**Cuándo usarlo:** para hacer mosaicos o duplicar un *batch* que luego se va a modificar.


In [29]:
x = torch.tensor([1, 2])
print(x.repeat(3))
print(x.repeat(2, 3))   # añade una dimensión

m = torch.tensor([[1, 2], [3, 4]])
print(m.repeat(1, 2))
print(m.repeat(2, 1))

tensor([1, 2, 1, 2, 1, 2])
tensor([[1, 2, 1, 2, 1, 2],
        [1, 2, 1, 2, 1, 2]])
tensor([[1, 2, 1, 2],
        [3, 4, 3, 4]])
tensor([[1, 2],
        [3, 4],
        [1, 2],
        [3, 4]])


## `x.repeat_interleave`

```python
x.repeat_interleave(repeats, dim=None)
```

Repite **cada elemento** de forma consecutiva: `[1, 2]` da `[1, 1, 2, 2]`, frente a `[1, 2, 1, 2]` de `repeat`.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `repeats` | Veces que se repite cada elemento. Puede ser un tensor con un valor por elemento. | obligatorio |
| `dim` | Dimensión. Sin indicarla, primero aplana. | aplana |

**Cuándo usarlo:** para *upsampling* por vecino más cercano: repetir en altura y anchura.


In [30]:
x = torch.tensor([1, 2])
print(x.repeat_interleave(2))
print(x.repeat_interleave(torch.tensor([1, 3])))

m = torch.tensor([[1, 2], [3, 4]])
print(m.repeat_interleave(2, dim=0))
print(m.repeat_interleave(2, dim=1))

tensor([1, 1, 2, 2])
tensor([1, 2, 2, 2])
tensor([[1, 2],
        [1, 2],
        [3, 4],
        [3, 4]])
tensor([[1, 1, 2, 2],
        [3, 3, 4, 4]])


## `x.expand / expand_as`

```python
x.expand(*sizes)
x.expand_as(other)
```

Amplía **sin copiar memoria**, y solo dimensiones de tamaño 1. Es mucho más barato que `repeat`.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `*sizes` | La forma final, no las repeticiones. `-1` significa «no cambiar esta dimensión». | obligatorio |
| `other` | Tensor cuya forma se copia. | obligatorio |

**Cuándo usarlo:** para el *token* CLS de ViT y para preparar los índices de `gather`.

**⚠️ Cuidado:** todas las copias comparten memoria, así que no escribas sobre el resultado sin hacer antes `.clone()`.


In [31]:
x = torch.tensor([[1], [2]])   # (2, 1)
print(x.expand(2, 3))
print(x.expand(-1, 3))
print(x.expand_as(torch.zeros(2, 4)))

# Token CLS de un ViT
cls_token = torch.zeros(1, 1, 8)
print(cls_token.expand(4, -1, -1).shape)

try:
    torch.zeros(2, 3).expand(4, 3)
except RuntimeError as error:
    print("Error:", str(error)[:60])

tensor([[1, 1, 1],
        [2, 2, 2]])
tensor([[1, 1, 1],
        [2, 2, 2]])
tensor([[1, 1, 1, 1],
        [2, 2, 2, 2]])
torch.Size([4, 1, 8])
Error: The expanded size of the tensor (4) must match the existing 


---

# 10. Mover datos

## `x.flip`

```python
x.flip(dims)
```

Invierte el orden en las dimensiones indicadas.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dims` | Dimensión o dimensiones que se invierten. | obligatorio |

**Cuándo usarlo:** para *data augmentation* y *test-time augmentation*. En imágenes, `-1` es el volteo horizontal.


In [32]:
m = torch.tensor([[1, 2], [3, 4]])
print(m.flip(-1))        # horizontal
print(m.flip(0))         # vertical
print(m.flip((0, 1)))    # ambas

tensor([[2, 1],
        [4, 3]])
tensor([[3, 4],
        [1, 2]])
tensor([[4, 3],
        [2, 1]])


## `torch.rot90`

```python
torch.rot90(input, k=1, dims=(0, 1))
```

Rota 90° `k` veces en el plano de las dos dimensiones indicadas.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `input` | El tensor. | obligatorio |
| `k` | Número de giros. Positivo: de `dims[0]` hacia `dims[1]`. | 1 |
| `dims` | Las dos dimensiones del plano de giro. | `(0, 1)` |

**Cuándo usarlo:** para *data augmentation*.


In [33]:
m = torch.tensor([[1, 2], [3, 4]])
print(torch.rot90(m))
print(torch.rot90(m, 2))
print(torch.rot90(m, -1))

image = torch.zeros(3, 4, 5)
print(torch.rot90(image, 1, dims=(-2, -1)).shape)

tensor([[2, 4],
        [1, 3]])
tensor([[4, 3],
        [2, 1]])
tensor([[3, 1],
        [4, 2]])
torch.Size([3, 5, 4])


## `torch.roll`

```python
torch.roll(input, shifts, dims=None)
```

Desplaza de forma **cíclica**: lo que sale por un lado entra por el otro.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `input` | El tensor. | obligatorio |
| `shifts` | Posiciones que se desplaza. Positivo: hacia índices mayores. | obligatorio |
| `dims` | Dimensiones. Sin indicarlas, aplana. | aplana |

**Cuándo usarlo:** para las ventanas desplazadas de Swin y las vibraciones de DeepDream.


In [34]:
x = torch.tensor([1, 2, 3, 4])
print(torch.roll(x, 1))
print(torch.roll(x, -1))

m = torch.tensor([[1, 2], [3, 4]])
print(torch.roll(m, 1, dims=1))
print(torch.roll(m, shifts=(1, 1), dims=(0, 1)))

tensor([4, 1, 2, 3])
tensor([2, 3, 4, 1])
tensor([[2, 1],
        [4, 3]])
tensor([[4, 3],
        [2, 1]])


## `x.unfold`

```python
x.unfold(dimension, size, step)
```

Extrae **ventanas deslizantes**. Añade al final una dimensión con el tamaño de la ventana.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dimension` | Dimensión sobre la que se desliza. | obligatorio |
| `size` | Tamaño de cada ventana. | obligatorio |
| `step` | Salto entre ventanas. Si es igual a `size`, no se solapan. | obligatorio |

**Cuándo usarlo:** para dividir audio en ventanas y para crear parches.


In [35]:
x = torch.arange(6)
print(x.unfold(0, 3, 1))
print(x.unfold(0, 3, 2))
print(x.unfold(0, 2, 2))

# Parches de una imagen
image = torch.zeros(3, 8, 8)
print(image.unfold(1, 4, 4).unfold(2, 4, 4).shape)

tensor([[0, 1, 2],
        [1, 2, 3],
        [2, 3, 4],
        [3, 4, 5]])
tensor([[0, 1, 2],
        [2, 3, 4]])
tensor([[0, 1],
        [2, 3],
        [4, 5]])
torch.Size([3, 2, 2, 4, 4])


---

# 11. Unir y separar

## `torch.cat`

```python
torch.cat(tensors, dim=0)
```

Concatena tensores por una dimensión **existente**. El resto de dimensiones debe coincidir.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `tensors` | Lista o tupla de tensores. | obligatorio |
| `dim` | Dimensión por la que se unen. | 0 |

**Cuándo usarlo:** para *skip connections* y para unir resultados por *batches*.

**⚠️ Cuidado:** recibe una **lista**: `torch.cat(a, b)` da error.


In [36]:
a, b = torch.zeros(2, 3), torch.ones(2, 3)
print(torch.cat([a, b]).shape)
print(torch.cat([a, b], dim=1).shape)

# Skip connection de una U-Net
up, skip = torch.zeros(1, 64, 8, 8), torch.zeros(1, 32, 8, 8)
print(torch.cat([up, skip], dim=1).shape)

torch.Size([4, 3])
torch.Size([2, 6])
torch.Size([1, 96, 8, 8])


## `torch.stack`

```python
torch.stack(tensors, dim=0)
```

Une tensores creando una dimensión **nueva**. Todos deben tener exactamente la misma forma.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `tensors` | Lista o tupla de tensores. | obligatorio |
| `dim` | Posición de la nueva dimensión. | 0 |

**Cuándo usarlo:** para formar un *batch* a partir de una lista y para recomponer cajas.


In [37]:
a, b = torch.zeros(3, 4), torch.ones(3, 4)
print(torch.stack([a, b]).shape)
print(torch.stack([a, b], dim=1).shape)
print(torch.stack([a, b], dim=-1).shape)

x1, y1, x2, y2 = torch.tensor([0.0]), torch.tensor([1.0]), torch.tensor([2.0]), torch.tensor([3.0])
print(torch.stack([x1, y1, x2, y2], dim=-1))

torch.Size([2, 3, 4])
torch.Size([3, 2, 4])
torch.Size([3, 4, 2])
tensor([[0., 1., 2., 3.]])


## `x.chunk / x.split`

```python
x.chunk(chunks, dim=0)
x.split(split_size_or_sections, dim=0)
```

Dividen un tensor y devuelven una tupla. `chunk` indica **cuántos trozos**; `split` indica **el tamaño** de cada uno.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `chunks` | Número de trozos. | obligatorio |
| `split_size_or_sections` | Un entero (tamaño de cada trozo) o una lista de tamaños. | obligatorio |
| `dim` | Dimensión que se divide. | 0 |

**Cuándo usarlo:** para separar la salida de una red: `mu, log_var = out.chunk(2, dim=1)`.


In [38]:
x = torch.arange(10)
print(x.chunk(2))
print(x.chunk(3))      # el último es más pequeño
print(x.split(4))
print(x.split([2, 8]))

b = torch.zeros(2, 6)
print([part.shape for part in b.split([1, 2, 3], dim=1)])

(tensor([0, 1, 2, 3, 4]), tensor([5, 6, 7, 8, 9]))
(tensor([0, 1, 2, 3]), tensor([4, 5, 6, 7]), tensor([8, 9]))
(tensor([0, 1, 2, 3]), tensor([4, 5, 6, 7]), tensor([8, 9]))
(tensor([0, 1]), tensor([2, 3, 4, 5, 6, 7, 8, 9]))
[torch.Size([2, 1]), torch.Size([2, 2]), torch.Size([2, 3])]


## `x.unbind`

```python
x.unbind(dim=0)
```

Separa por una dimensión y **elimina** esa dimensión. Es lo contrario de `stack`.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dim` | Dimensión que se separa. | 0 |

**Cuándo usarlo:** para desempaquetar las coordenadas de una caja.


In [39]:
boxes = torch.tensor([[1.0, 2.0, 3.0, 4.0], [5.0, 6.0, 7.0, 8.0]])
x1, y1, x2, y2 = boxes.unbind(-1)
print(x1, y2)
print([row.shape for row in boxes.unbind(0)])

tensor([1., 5.]) tensor([4., 8.])
[torch.Size([4]), torch.Size([4])]


---

# 12. Indexar y seleccionar

## `Indexado con corchetes`

```python
x[i]   x[:, j]   x[..., -1]   x[None]   x[mask]   x[indices]
```

`:` selecciona todo, `...` sustituye a varias dimensiones, y `None` añade una. Con **slices**, el resultado comparte memoria. Con máscaras o listas de índices, es una copia.

**Cuándo usarlo:** en todo momento.

**⚠️ Cuidado:** una máscara devuelve siempre un tensor 1D, se pierde la forma.


In [40]:
x = torch.arange(12).reshape(3, 4)
print(x)
print(x[0])            # primera fila
print(x[:, 1])         # segunda columna
print(x[1, 2])         # un elemento
print(x[0:2, 1:3])     # submatriz
print(x[..., -1])      # última dimensión
print(x[::2])          # una fila de cada dos
print(x[[0, 2]])       # filas 0 y 2
print(x[x > 8])        # con máscara: resultado 1D

# Un elemento por fila
print(x[torch.arange(3), torch.tensor([0, 1, 2])])

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([0, 1, 2, 3])
tensor([1, 5, 9])
tensor(6)
tensor([[1, 2],
        [5, 6]])
tensor([ 3,  7, 11])
tensor([[ 0,  1,  2,  3],
        [ 8,  9, 10, 11]])
tensor([[ 0,  1,  2,  3],
        [ 8,  9, 10, 11]])
tensor([ 9, 10, 11])
tensor([ 0,  5, 10])


## `x.index_select`

```python
x.index_select(dim, index)
```

Selecciona posiciones completas de una dimensión.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dim` | Dimensión en la que se selecciona. | obligatorio |
| `index` | Tensor 1D de tipo `int64`. | obligatorio |

**Cuándo usarlo:** para quedarte con las cajas que sobreviven a NMS.


In [41]:
x = torch.arange(12).reshape(3, 4)
print(x.index_select(0, torch.tensor([2, 0])))
print(x.index_select(1, torch.tensor([0, 3])))

tensor([[ 8,  9, 10, 11],
        [ 0,  1,  2,  3]])
tensor([[ 0,  3],
        [ 4,  7],
        [ 8, 11]])


## `torch.where`

```python
torch.where(condition, input, other)
torch.where(condition)
```

Con tres argumentos, **elige elemento a elemento**: de `input` si la condición es `True`, y de `other` si no. Con uno, equivale a buscar las posiciones a `True`.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `condition` | Máscara booleana. | obligatorio |
| `input` | Valores si es `True`. Tensor o número. | — |
| `other` | Valores si es `False`. Tensor o número. | — |

**Cuándo usarlo:** para condiciones por elemento sin bucles, como la Smooth L1.


In [42]:
x = torch.tensor([-1.0, 2.0, -3.0])
print(torch.where(x > 0, x, 0.0))
print(torch.where(x > 0, 1, -1))
print(torch.where(x > 0))

rows, cols = torch.where(torch.tensor([[True, False], [False, True]]))
print(rows, cols)

tensor([0., 2., 0.])
tensor([-1,  1, -1])
(tensor([1]),)
tensor([0, 1]) tensor([0, 1])


## `x.masked_fill / masked_select`

```python
x.masked_fill(mask, value)
x.masked_select(mask)
```

`masked_fill` escribe un valor donde la máscara es `True` y conserva la forma. `masked_select` extrae esos valores en un tensor 1D.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `mask` | Máscara booleana difundible a `x`. | obligatorio |
| `value` | Valor de relleno. | obligatorio |

**Cuándo usarlo:** `masked_fill` con `-inf` antes del softmax de atención.

**⚠️ Cuidado:** `masked_fill_`, con guion bajo, modifica el tensor original.


In [43]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
mask = x > 2
print(x.masked_fill(mask, -1.0))
print(x.masked_select(mask))
print(x)   # no ha cambiado

# Máscara de atención
scores = torch.zeros(2, 2)
causal = torch.tril(torch.ones(2, 2, dtype=torch.bool))
print(scores.masked_fill(~causal, float("-inf")))

tensor([[ 1.,  2.],
        [-1., -1.]])
tensor([3., 4.])
tensor([[1., 2.],
        [3., 4.]])
tensor([[0., -inf],
        [0., 0.]])


## `x.nonzero`

```python
x.nonzero(as_tuple=False)
```

Devuelve las coordenadas de los elementos distintos de 0 o `True`.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `as_tuple` | Si es `True`, devuelve una tupla con los índices de cada dimensión. | `False` |

**Cuándo usarlo:** para encontrar las celdas con objeto en YOLO o los píxeles de una máscara.


In [44]:
mask = torch.tensor([[False, True], [True, True]])
print(mask.nonzero())
print(mask.nonzero(as_tuple=True))

tensor([[0, 1],
        [1, 0],
        [1, 1]])
(tensor([0, 1, 1]), tensor([1, 0, 1]))


## `x.gather`

```python
x.gather(dim, index)
```

Toma elementos usando un tensor de índices. `index` y el resultado tienen **la misma forma**. Para `dim=1`: `out[i][j] = x[i][index[i][j]]`.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dim` | Dimensión a la que se refieren los índices. | obligatorio |
| `index` | Índices `int64`, con el mismo número de dimensiones que `x`. | obligatorio |

**Cuándo usarlo:** para tomar la puntuación de la clase correcta o la caja responsable en YOLO.

**⚠️ Cuidado:** `index` debe tener el mismo número de dimensiones: de ahí el `unsqueeze(1)`.


In [45]:
logits = torch.tensor([[0.1, 0.7, 0.2], [0.8, 0.1, 0.1]])
labels = torch.tensor([1, 0])
print(logits.gather(1, labels.unsqueeze(1)))
print(logits.gather(1, labels.unsqueeze(1)).squeeze(1))

index = torch.tensor([[2, 0], [1, 1]])
print(logits.gather(1, index))

tensor([[0.7000],
        [0.8000]])
tensor([0.7000, 0.8000])
tensor([[0.2000, 0.1000],
        [0.1000, 0.1000]])


## `x.scatter / scatter_ / scatter_add_`

```python
x.scatter(dim, index, src)
x.scatter_(dim, index, src)
x.scatter_add_(dim, index, src)
```

Lo contrario de `gather`: **escribe** en las posiciones indicadas. `scatter_add_` suma en lugar de sustituir.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dim` | Dimensión a la que se refieren los índices. | obligatorio |
| `index` | Posiciones `int64`. | obligatorio |
| `src` | Valor o tensor con lo que se escribe. | obligatorio |

**Cuándo usarlo:** para crear *one-hot* y contar píxeles por clase.


In [46]:
labels = torch.tensor([2, 0])
print(torch.zeros(2, 3).scatter(1, labels.unsqueeze(1), 1.0))

# Histograma
values = torch.tensor([0, 2, 2, 1, 2])
counts = torch.zeros(3).scatter_add_(0, values, torch.ones(5))
print(counts)

tensor([[0., 0., 1.],
        [1., 0., 0.]])
tensor([1., 1., 3.])


## `torch.tril / triu / diag`

```python
torch.tril(input, diagonal=0)
torch.triu(input, diagonal=0)
torch.diag(input, diagonal=0)
```

`tril` conserva la mitad **inferior** y `triu`, la **superior**. `diag` crea una matriz diagonal desde un vector, o extrae la diagonal de una matriz.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `input` | Vector o matriz. | obligatorio |
| `diagonal` | Qué diagonal: 0 es la principal; positivo, por encima; negativo, por debajo. | 0 |

**Cuándo usarlo:** para las máscaras causales de atención y PixelCNN.


In [47]:
m = torch.ones(3, 3, dtype=torch.int64)
print(torch.tril(m))
print(torch.triu(m, diagonal=1))
print(torch.diag(torch.tensor([1, 2])))
print(torch.diag(torch.tensor([[1, 2], [3, 4]])))

tensor([[1, 0, 0],
        [1, 1, 0],
        [1, 1, 1]])
tensor([[0, 1, 1],
        [0, 0, 1],
        [0, 0, 0]])
tensor([[1, 0],
        [0, 2]])
tensor([1, 4])


## `torch.meshgrid`

```python
torch.meshgrid(*tensors, indexing="ij")
```

A partir de vectores de coordenadas, crea **rejillas**, una por vector.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `*tensors` | Vectores 1D. | obligatorio |
| `indexing` | `"ij"`: el primer vector corresponde a las filas. `"xy"`: el primero, a las columnas. | pásalo siempre |

**Cuándo usarlo:** para generar anchors, ejes de coordenadas y centros de celda de YOLO.


In [48]:
rows, cols = torch.meshgrid(torch.arange(2), torch.arange(3), indexing="ij")
print(rows)
print(cols)
print(torch.stack([cols, rows], dim=-1).shape)

tensor([[0, 0, 0],
        [1, 1, 1]])
tensor([[0, 1, 2],
        [0, 1, 2]])
torch.Size([2, 3, 2])


---

# 13. Reducciones

Eliminan una o varias dimensiones al resumirlas en un valor.

## `x.sum / mean / prod`

```python
x.sum(dim=None, keepdim=False)
x.mean(dim=None, keepdim=False)
x.prod(dim=None, keepdim=False)
```

`dim` es la dimensión que **desaparece**. Sin `dim`, se reducen todas a un escalar. `mean` necesita un tipo `float`.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dim` | Dimensión sobre la que se opera. Admite negativos: `-1` es la última. | todas |
| `keepdim` | Si es `True`, la dimensión reducida se conserva con tamaño 1, lo que permite hacer *broadcasting* con el original. | `False` |

**Cuándo usarlo:** `dim=(2, 3)` es un *Global Average Pooling*; y `(pred == y).float().mean()`, la *accuracy*.

**⚠️ Cuidado:** `dim=0` no significa «la fila 0», sino que colapsa las filas y deja un valor por columna.


In [49]:
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(x.sum())
print(x.sum(dim=0))         # por columnas
print(x.sum(dim=1))         # por filas
print(x.mean(dim=1, keepdim=True))
print(x.prod())

images = torch.rand(2, 3, 4, 4)
print(images.mean(dim=(2, 3)).shape)
print(images.mean(dim=(2, 3), keepdim=True).shape)

mask = torch.tensor([True, True, False])
print(mask.sum(), mask.float().mean())   # conteo y fracción

tensor(21.)
tensor([5., 7., 9.])
tensor([ 6., 15.])
tensor([[2.],
        [5.]])
tensor(720.)
torch.Size([2, 3])
torch.Size([2, 3, 1, 1])
tensor(2) tensor(0.6667)


## `x.std / x.var / x.median`

```python
x.std(dim=None, keepdim=False, unbiased=True)
x.var(dim=None, keepdim=False, unbiased=True)
x.median(dim=None, keepdim=False)
```

Desviación típica, varianza y mediana. Por defecto, `std` y `var` dividen entre `n - 1`.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dim` | Dimensión sobre la que se opera. Admite negativos: `-1` es la última. | todas |
| `keepdim` | Si es `True`, la dimensión reducida se conserva con tamaño 1, lo que permite hacer *broadcasting* con el original. | `False` |
| `unbiased` | `True` divide entre `n - 1`; `False`, entre `n`. | `True` |

**Cuándo usarlo:** para normalizar datos: `(x - x.mean()) / x.std()`.

**⚠️ Cuidado:** `median` con `dim` devuelve `(valores, índices)`.


In [50]:
x = torch.tensor([1.0, 2.0, 3.0, 4.0])
print(x.std(), x.std(unbiased=False))
print(x.var())
print(x.median())   # con dos centrales, toma el menor

m = torch.tensor([[1.0, 9.0], [3.0, 5.0]])
print(m.median(dim=0))

tensor(1.2910) tensor(1.1180)
tensor(1.6667)
tensor(2.)
torch.return_types.median(
values=tensor([1., 5.]),
indices=tensor([0, 1]))


## `x.max / x.min`

```python
x.max()
x.max(dim, keepdim=False)
torch.max(a, b)
```

Sin `dim`, el máximo global. **Con `dim`, devuelve dos cosas: `(valores, índices)`**. `min` funciona igual.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dim` | Dimensión sobre la que se opera. Admite negativos: `-1` es la última. | todas |
| `keepdim` | Si es `True`, la dimensión reducida se conserva con tamaño 1, lo que permite hacer *broadcasting* con el original. | `False` |

**Cuándo usarlo:** para quedarte con la puntuación máxima y su clase.

**⚠️ Cuidado:** `x.max(dim=1) + 1` falla, porque devuelve una estructura con dos tensores.


In [51]:
x = torch.tensor([[1.0, 5.0], [7.0, 2.0]])
print(x.max())
values, indices = x.max(dim=1)
print(values, indices)
print(x.max(dim=1).values)
print(torch.max(x, torch.full_like(x, 3.0)))   # elemento a elemento

tensor(7.)
tensor([5., 7.]) tensor([1, 0])
tensor([5., 7.])
tensor([[3., 5.],
        [7., 3.]])


## `x.amax / x.amin`

```python
x.amax(dim=(), keepdim=False)
x.amin(dim=(), keepdim=False)
```

Como `max`/`min`, pero **solo devuelven valores** y admiten varias dimensiones.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dim` | Dimensión o tupla de dimensiones. | todas |
| `keepdim` | Si es `True`, la dimensión reducida se conserva con tamaño 1, lo que permite hacer *broadcasting* con el original. | `False` |

**Cuándo usarlo:** para normalizar mapas de explicabilidad por imagen.


In [52]:
images = torch.rand(2, 3, 4, 4)
low = images.amin(dim=(1, 2, 3), keepdim=True)
high = images.amax(dim=(1, 2, 3), keepdim=True)
normalized = (images - low) / (high - low)
print(low.shape)
print(normalized.amin(dim=(1, 2, 3)), normalized.amax(dim=(1, 2, 3)))

torch.Size([2, 1, 1, 1])
tensor([0., 0.]) tensor([1., 1.])


## `x.argmax / x.argmin`

```python
x.argmax(dim=None, keepdim=False)
x.argmin(dim=None, keepdim=False)
```

Devuelven la **posición** del máximo o del mínimo. Sin `dim`, el índice es sobre el tensor aplanado.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dim` | Dimensión sobre la que se opera. Admite negativos: `-1` es la última. | todas |
| `keepdim` | Si es `True`, la dimensión reducida se conserva con tamaño 1, lo que permite hacer *broadcasting* con el original. | `False` |

**Cuándo usarlo:** para pasar de *logits* a la clase predicha. En segmentación: `out.argmax(dim=1)`.


In [53]:
logits = torch.tensor([[0.1, 2.0, 0.3], [5.0, 1.0, 0.0]])
print(logits.argmax())
print(logits.argmax(dim=1))

labels = torch.tensor([1, 2])
print((logits.argmax(dim=1) == labels).float().mean())

print(divmod(logits.argmax().item(), logits.shape[1]))   # fila y columna

tensor(3)
tensor([1, 0])
tensor(0.5000)
(1, 0)


## `x.cumsum / x.cumprod`

```python
x.cumsum(dim)
x.cumprod(dim)
```

Suma o producto **acumulado**. No reducen: conservan la forma.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dim` | Dimensión del acumulado. **Es obligatoria**. | obligatorio |

**Cuándo usarlo:** para las curvas PR del mAP y el *beta* acumulado de difusión.


In [54]:
x = torch.tensor([1, 2, 3, 4])
print(x.cumsum(0))
print(x.cumprod(0))

# Curva precisión-recall
tp = torch.tensor([1.0, 0.0, 1.0, 1.0])
tp_acc = tp.cumsum(0)
print(tp_acc / torch.arange(1, 5))   # precisión
print(tp_acc / 3)                     # recall

tensor([ 1,  3,  6, 10])
tensor([ 1,  2,  6, 24])
tensor([1.0000, 0.5000, 0.6667, 0.7500])
tensor([0.3333, 0.3333, 0.6667, 1.0000])


## `x.norm`

```python
x.norm(p="fro", dim=None, keepdim=False)
```

Calcula la norma. Por defecto, ℓ2 sobre todos los elementos. `torch.linalg.vector_norm` es la versión moderna.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `p` | Tipo de norma: `2`, `1` o `float("inf")`. | ℓ2 |
| `dim` | Dimensión sobre la que se opera. Admite negativos: `-1` es la última. | todas |
| `keepdim` | Si es `True`, la dimensión reducida se conserva con tamaño 1, lo que permite hacer *broadcasting* con el original. | `False` |

**Cuándo usarlo:** para normalizar vectores y medir perturbaciones adversarias.


In [55]:
x = torch.tensor([[3.0, 4.0], [6.0, 8.0]])
print(x.norm())
print(x.norm(dim=1))
print(x.norm(p=1, dim=1))
print(x.norm(p=float("inf"), dim=1))
print(x / x.norm(dim=1, keepdim=True))

tensor(11.1803)
tensor([ 5., 10.])
tensor([ 7., 14.])
tensor([4., 8.])
tensor([[0.6000, 0.8000],
        [0.6000, 0.8000]])


---

# 14. Ordenar y contar

## `x.sort / x.argsort`

```python
x.sort(dim=-1, descending=False)
x.argsort(dim=-1, descending=False)
```

`sort` devuelve `(valores, índices)`; `argsort`, solo los índices.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dim` | Dimensión que se ordena. | -1 |
| `descending` | `True`: de mayor a menor. | `False` |

**Cuándo usarlo:** para ordenar cajas en NMS y píxeles por importancia en loyalty.


In [56]:
x = torch.tensor([3.0, 1.0, 2.0])
values, indices = x.sort(descending=True)
print(values, indices)
print(x.argsort())

boxes = torch.tensor([[0.0], [1.0], [2.0]])
print(boxes[x.argsort(descending=True)])

tensor([3., 2., 1.]) tensor([0, 2, 1])
tensor([1, 2, 0])
tensor([[0.],
        [2.],
        [1.]])


## `x.topk`

```python
x.topk(k, dim=-1, largest=True, sorted=True)
```

Devuelve los `k` mayores como `(valores, índices)`. Es más eficiente que ordenar todo.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `k` | Cuántos elementos. | obligatorio |
| `dim` | Dimensión. | -1 |
| `largest` | `False` devuelve los `k` menores. | `True` |
| `sorted` | Si el resultado se devuelve ordenado. | `True` |

**Cuándo usarlo:** para calcular *top-k accuracy* o quedarte con los parches más activos.


In [57]:
x = torch.tensor([1.0, 5.0, 3.0, 4.0])
print(x.topk(2))
print(x.topk(2, largest=False))

logits = torch.tensor([[0.1, 0.5, 0.4], [0.7, 0.2, 0.1]])
print(logits.topk(2, dim=1).indices)

torch.return_types.topk(
values=tensor([5., 4.]),
indices=tensor([1, 3]))
torch.return_types.topk(
values=tensor([1., 3.]),
indices=tensor([0, 2]))
tensor([[1, 2],
        [0, 1]])


## `torch.unique`

```python
torch.unique(input, sorted=True, return_inverse=False, return_counts=False, dim=None)
```

Devuelve los **valores distintos**.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `input` | El tensor. | obligatorio |
| `sorted` | Si se devuelven ordenados. | `True` |
| `return_inverse` | Devuelve además, por cada elemento, su posición en la lista de valores únicos. | `False` |
| `return_counts` | Devuelve además cuántas veces aparece cada uno. | `False` |
| `dim` | Busca filas o columnas únicas. | aplana |

**Cuándo usarlo:** para ver las clases de una máscara de segmentación.


In [58]:
x = torch.tensor([3, 1, 3, 3])
print(torch.unique(x))
print(torch.unique(x, return_counts=True))
print(torch.unique(x, return_inverse=True))

mask = torch.tensor([[0, 0], [2, 2]])
print(mask.unique())

tensor([1, 3])
(tensor([1, 3]), tensor([1, 3]))
(tensor([1, 3]), tensor([1, 0, 1, 1]))
tensor([0, 2])


## `torch.bincount`

```python
torch.bincount(input, weights=None, minlength=0)
```

Cuenta cuántas veces aparece cada entero. El índice del resultado es el valor.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `input` | Tensor 1D de enteros no negativos. | obligatorio |
| `weights` | Suma estos pesos en lugar de sumar 1. | `None` |
| `minlength` | Tamaño mínimo del resultado; sirve para incluir las clases ausentes. | 0 |

**Cuándo usarlo:** para histogramas de clases y matrices de confusión.

**⚠️ Cuidado:** sin `minlength`, las clases finales que no aparecen no se incluyen, y el tamaño puede variar.


In [59]:
labels = torch.tensor([0, 0, 2])
print(torch.bincount(labels))
print(torch.bincount(labels, minlength=4))
print(torch.bincount(labels, weights=torch.tensor([0.5, 0.5, 2.0])))

tensor([2, 0, 1])
tensor([2, 0, 1, 0])
tensor([1., 0., 2.])


## `x.any / x.all`

```python
x.any(dim=None, keepdim=False)
x.all(dim=None, keepdim=False)
```

`any`: ¿hay **algún** `True`? `all`: ¿son **todos** `True`?

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `dim` | Dimensión sobre la que se opera. Admite negativos: `-1` es la última. | todas |
| `keepdim` | Si es `True`, la dimensión reducida se conserva con tamaño 1, lo que permite hacer *broadcasting* con el original. | `False` |

**Cuándo usarlo:** para hallar filas con algún píxel en una máscara, y en comprobaciones como `(x >= 0).all()`.


In [60]:
mask = torch.tensor([[True, True], [True, False], [False, False]])
print(mask.any(), mask.all())
print(mask.any(dim=1))
print(mask.all(dim=1))

tensor(True) tensor(False)
tensor([ True,  True, False])
tensor([ True, False, False])


---

# 15. Matemáticas y comparaciones

## `x.clamp`

```python
x.clamp(min=None, max=None)
x.clamp_min(min)
x.clamp_max(max)
```

Recorta los valores a un rango. `min` y `max` pueden ser números o tensores.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `min` | Límite inferior. | sin límite |
| `max` | Límite superior. | sin límite |

**Cuándo usarlo:** tras un ataque, con `clamp(0, 1)`; también para la intersección del IoU y para evitar divisiones entre cero.


In [61]:
x = torch.tensor([-1.0, 0.5, 2.0])
print(x.clamp(0, 1))
print(x.clamp(min=0))
print(x.clamp_min(0.1))

boxes = torch.tensor([[-5.0, 3.0, 40.0, 50.0]])
print(boxes.clamp(min=torch.zeros(4), max=torch.tensor([30.0, 20.0, 30.0, 20.0])))

# Evita dividir entre cero
print(1 / torch.tensor([0.0]).clamp_min(1e-8))

tensor([0.0000, 0.5000, 1.0000])
tensor([0.0000, 0.5000, 2.0000])
tensor([0.1000, 0.5000, 2.0000])
tensor([[ 0.,  3., 30., 20.]])
tensor([100000000.])


## `Funciones por elemento`

```python
x.abs()   x.sqrt()   x.exp()   x.log()   x.sign()   x.pow(n)
x.round()   x.floor()   x.ceil()
```

Se aplican a cada elemento y conservan la forma.

**Cuándo usarlo:** `sign` en FGSM; `sqrt` en YOLO; `exp` en VAE.

**⚠️ Cuidado:** `log` de 0 da `-inf`, y `sqrt` o `log` de un negativo dan `nan`.


In [62]:
x = torch.tensor([-1.5, 0.25, 4.0])
print(x.abs())
print(x.sign())
print(x.round(), x.floor(), x.ceil())
print(x.pow(2), x ** 2)
print(x.clamp_min(0).sqrt())
print(x.exp())
print(x.clamp_min(1e-8).log())

tensor([1.5000, 0.2500, 4.0000])
tensor([-1.,  1.,  1.])
tensor([-2.,  0.,  4.]) tensor([-2.,  0.,  4.]) tensor([-1.,  1.,  4.])
tensor([ 2.2500,  0.0625, 16.0000]) tensor([ 2.2500,  0.0625, 16.0000])
tensor([0.0000, 0.5000, 2.0000])
tensor([ 0.2231,  1.2840, 54.5981])
tensor([-18.4207,  -1.3863,   1.3863])


## `torch.maximum / torch.minimum`

```python
torch.maximum(input, other)
torch.minimum(input, other)
```

Comparan **elemento a elemento** dos tensores. Es distinto de `amax`, que reduce.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `input` | Primer tensor. | obligatorio |
| `other` | Segundo tensor, difundible. | obligatorio |

**Cuándo usarlo:** para el IoU.


In [63]:
a = torch.tensor([1.0, 5.0])
b = torch.tensor([3.0, 2.0])
print(torch.maximum(a, b), torch.minimum(a, b))

# Intersección de dos cajas
box1, box2 = torch.tensor([0.0, 0, 2, 2]), torch.tensor([1.0, 1, 3, 3])
top_left = torch.maximum(box1[:2], box2[:2])
bottom_right = torch.minimum(box1[2:], box2[2:])
print((bottom_right - top_left).clamp(min=0).prod())

tensor([3., 5.]) tensor([1., 2.])
tensor(1.)


## `torch.lerp`

```python
torch.lerp(input, end, weight)
```

Interpolación lineal: `input + weight * (end - input)`.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `input` | Valor de partida. | obligatorio |
| `end` | Valor final. | obligatorio |
| `weight` | Peso: número o tensor. 0 da `input`; 1 da `end`. | obligatorio |

**Cuándo usarlo:** para Mixup, Integrated Gradients y la media exponencial de pesos.


In [64]:
a, b = torch.tensor([0.0, 10.0]), torch.tensor([10.0, 20.0])
print(torch.lerp(a, b, 0.25))
print(torch.lerp(a, b, torch.tensor([0.0, 1.0])))

tensor([ 2.5000, 12.5000])
tensor([ 0., 20.])


## `Comparaciones`

```python
x == y   x > y   x != y
torch.equal(a, b)
torch.allclose(a, b, rtol=1e-5, atol=1e-8)
x.isnan()   x.isinf()
```

Los operadores comparan **elemento a elemento** y dan una máscara. `equal` da un único booleano: misma forma y valores. `allclose` admite tolerancia. Es la forma correcta de comparar decimales.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `rtol` | Tolerancia relativa. | `1e-5` |
| `atol` | Tolerancia absoluta. | `1e-8` |

**Cuándo usarlo:** para depurar con `isnan` y para los tests.

**⚠️ Cuidado:** para combinar máscaras usa `&`, `|` y `~`, con paréntesis, y no `and` ni `or`.


In [65]:
a, b = torch.tensor([1.0, 2.0]), torch.tensor([1.0, 3.0])
print(a == b, a < b)
print(torch.equal(a, a), torch.equal(a, b))
print(0.1 + 0.2 == 0.3)
print(torch.allclose(torch.tensor(0.1 + 0.2), torch.tensor(0.3)))

x = torch.tensor([1.0, float("nan"), float("inf")])
print(x.isnan(), x.isinf())

# Combinar máscaras
y = torch.tensor([1, 5, 9])
print((y > 2) & (y < 8), (y < 2) | (y > 8), ~(y > 2))

tensor([ True, False]) tensor([False,  True])
True False
False
True
tensor([False,  True, False]) tensor([False, False,  True])
tensor([False,  True, False]) tensor([ True, False,  True]) tensor([ True, False, False])


---

# 16. Broadcasting

Permite operar con formas distintas. Se **alinean desde la derecha**, y cada dimensión debe coincidir, valer 1 o no existir.

## `Reglas de broadcasting`

```python
torch.broadcast_to(input, shape)
```

Ninguna operación necesita indicarlo: `+`, `*`, etc. aplican estas reglas automáticamente. `broadcast_to` lo hace explícito.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `input` | Tensor. | obligatorio |
| `shape` | Forma final. | obligatorio |

**Cuándo usarlo:** para IoU, distancias por parejas y normalización por canal.

**⚠️ Cuidado:** `(N, 1)` con `(N,)` no da error, sino `(N, N)`. Es un fallo silencioso muy habitual al calcular pérdidas.


In [66]:
print((torch.zeros(3, 4) + torch.zeros(4)).shape)
print((torch.zeros(5, 1, 4) + torch.zeros(3, 4)).shape)

# Todas las parejas de cajas
boxes1, boxes2 = torch.zeros(5, 4), torch.zeros(3, 4)
print((boxes1[:, None, :] - boxes2[None, :, :]).shape)

try:
    torch.zeros(3) + torch.zeros(4)
except RuntimeError as error:
    print("Error:", str(error)[:58])

print(torch.broadcast_to(torch.tensor([1, 2]), (2, 2)))

# Trampa habitual: (N, 1) y (N) dan (N, N)
print((torch.zeros(3, 1) - torch.zeros(3)).shape)

torch.Size([3, 4])
torch.Size([5, 3, 4])
torch.Size([5, 3, 4])
Error: The size of tensor a (3) must match the size of tensor b (
tensor([[1, 2],
        [1, 2]])
torch.Size([3, 3])


---

# 17. Álgebra lineal

## `@ / torch.matmul / torch.bmm / torch.dot`

```python
a @ b
torch.matmul(a, b)
torch.bmm(a, b)
torch.dot(a, b)
```

`@` y `matmul` son el producto matricial, con *batch* en las dimensiones iniciales. `bmm` exige exactamente 3 dimensiones. `dot` es el producto escalar de dos vectores 1D. `*` **no** es el producto matricial, sino elemento a elemento.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `a, b` | Operandos. Las dos últimas dimensiones deben cumplir `(n, m)` y `(m, k)`. | obligatorio |

**Cuándo usarlo:** en atención (`q @ k.transpose(-2, -1)`) y en la matriz de Gram.


In [67]:
a, b = torch.rand(2, 3), torch.rand(3, 4)
print((a @ b).shape)
print(torch.bmm(torch.rand(5, 2, 3), torch.rand(5, 3, 4)).shape)
print(torch.dot(torch.tensor([1.0, 2.0]), torch.tensor([3.0, 4.0])))

m = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(m * m)
print(m @ m)

# Transpuestas
print(m.T)
print(torch.rand(5, 2, 3).mT.shape)   # traspone las dos últimas

torch.Size([2, 4])
torch.Size([5, 2, 4])
tensor(11.)
tensor([[ 1.,  4.],
        [ 9., 16.]])
tensor([[ 7., 10.],
        [15., 22.]])
tensor([[1., 3.],
        [2., 4.]])
torch.Size([5, 3, 2])


## `torch.einsum`

```python
torch.einsum(equation, *operands)
```

Describe la operación con letras. Una letra repetida entre operandos se **multiplica**; una que no aparece a la derecha de `->` se **suma**.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `equation` | Texto como `"ij,jk->ik"`: letras de cada entrada y, tras `->`, de la salida. | obligatorio |
| `*operands` | Los tensores. | obligatorio |

**Cuándo usarlo:** cuando los `transpose` y `bmm` son difíciles de leer: Gram y atención.


In [68]:
a, b = torch.rand(2, 3), torch.rand(3, 4)
print(torch.einsum("ij,jk->ik", a, b).shape)      # producto matricial
print(torch.einsum("ij->ji", a).shape)             # transpuesta
print(torch.einsum("ij->", a), a.sum())            # suma total
print(torch.einsum("ii->", torch.eye(3)))          # traza

# Matriz de Gram por batch
features = torch.rand(2, 3, 16)   # (B, C, H*W)
print(torch.einsum("bci,bdi->bcd", features, features).shape)

torch.Size([2, 4])
torch.Size([3, 2])
tensor(2.9059) tensor(2.9059)
tensor(3.)
torch.Size([2, 3, 3])


## `torch.cdist`

```python
torch.cdist(x1, x2, p=2.0)
```

Distancia entre **todas las parejas** de dos conjuntos de vectores.

| Argumento | Qué significa | Por defecto |
|---|---|---|
| `x1` | Forma `(..., N, D)`. | obligatorio |
| `x2` | Forma `(..., M, D)`. | obligatorio |
| `p` | Tipo de norma: 2 es la euclídea. | 2.0 |

**Cuándo usarlo:** para la distancia de Chamfer y la cuantización de VQ-VAE.


In [69]:
a = torch.tensor([[0.0, 0.0], [1.0, 1.0]])
b = torch.tensor([[3.0, 4.0]])
print(torch.cdist(a, b))
print(torch.cdist(a, b, p=1))

tensor([[5.0000],
        [3.6056]])
tensor([[7.],
        [5.]])


---

# 18. Memoria

Muchos errores silenciosos vienen de aquí.

## `x.clone / x.detach`

```python
x.clone()
x.detach()
```

`clone` **copia los datos** y sigue en el grafo. `detach` **saca del grafo**, pero **comparte memoria**. `x.detach().clone()` hace ambas cosas.

**Cuándo usarlo:** `fake.detach()` al entrenar el discriminador; `x.detach().clone()` al iniciar un ataque adversario.


In [70]:
x = torch.ones(2, requires_grad=True)
cloned = x.clone()
detached = x.detach()
print(cloned.requires_grad, detached.requires_grad)

detached[0] = 99
print(x)   # detach comparte memoria

x = torch.ones(2, requires_grad=True)
safe = x.detach().clone()
safe[0] = 99
print(x)   # no cambia

True False
tensor([99.,  1.], requires_grad=True)
tensor([1., 1.], requires_grad=True)


## `Operaciones en el sitio: el guion bajo`

```python
x.add_(1)   x.mul_(2)   x.zero_()   x.clamp_(0, 1)   x.uniform_(a, b)
```

Un método con `_` final **modifica el propio tensor**. Sin guion bajo, devuelve uno nuevo.

**Cuándo usarlo:** para ahorrar memoria y dentro de `torch.no_grad()` para modificar pesos.

**⚠️ Cuidado:** una operación en el sitio sobre un tensor hoja con `requires_grad=True` da error.


In [71]:
x = torch.ones(2)
result = x.add(1)
print(x, result)
x.add_(1)
print(x)

x.zero_()
print(x)

print(torch.empty(2).uniform_(0, 1))
print(torch.empty(2).normal_(0, 1))

leaf = torch.ones(1, requires_grad=True)
try:
    leaf.add_(1)
except RuntimeError as error:
    print("Error:", str(error)[:50])
with torch.no_grad():
    leaf.add_(1)
print(leaf)

tensor([1., 1.]) tensor([2., 2.])
tensor([2., 2.])
tensor([0., 0.])
tensor([0.3577, 0.0860])
tensor([-0.9145, -1.0704])
Error: a leaf Variable that requires grad is being used i
tensor([2.], requires_grad=True)


## `Qué comparte memoria`

```python
x.view   x.permute   x.expand   x[0]   x.detach   torch.from_numpy
```

Las **vistas** no copian datos: escribir en ellas cambia el original. `clone`, `repeat`, `cat`, `stack`, `torch.tensor` y el indexado con máscara **sí** copian.

**Cuándo usarlo:** ante cualquier duda, usa `.clone()`.


In [72]:
x = torch.zeros(3)
x.view(3, 1)[0, 0] = 5
print("view:", x)

x = torch.zeros(3)
x[1:][0] = 5
print("slice:", x)

x = torch.zeros(3)
x.clone()[0] = 5
print("clone:", x)

x = torch.zeros(3)
x[x == 0][0] = 5
print("máscara al leer:", x)

x = torch.zeros(3)
x[x == 0] = 5
print("máscara al escribir:", x)

view: tensor([5., 0., 0.])
slice: tensor([0., 5., 0.])
clone: tensor([0., 0., 0.])
máscara al leer: tensor([0., 0., 0.])
máscara al escribir: tensor([5., 5., 5.])
